# 07 - Interactive Mapping in Jupyter: Capturing Clicks with iPyleaflet

## Overview

Earlier in the course we used **Folium** to create interactive maps. Folium is excellent for displaying spatial data and exporting polished maps. However, when we want **Python to respond to user interaction**, such as clicking on the map and saving coordinates, we need a different tool.

In this lesson we switch to **ipyleaflet**, which integrates directly with Jupyter notebooks and allows Python to react to map events like mouse clicks.

Using ipyleaflet, we will build a small interactive tool that:

- Displays a map
- Captures latitude and longitude when the user clicks
- Places a marker on the clicked location
- Saves the coordinates to a file
- Provides a button to clear saved points

This demonstrates how spatial visualization and user interaction can work together inside a notebook environment.


## Why Move from Folium to ipyleaflet?

### Folium

Folium is designed primarily for **map visualization**.

It is great for:

- adding markers and popups
- displaying layers
- creating choropleths
- exporting standalone HTML maps

But once the map is rendered, most interaction happens **inside JavaScript in the browser**, not in Python.

That means Python cannot easily respond to events like mouse clicks.

### ipyleaflet

ipyleaflet solves this problem.

It allows:

- Python to receive map click events
- dynamic updates to the map
- integration with Jupyter widgets
- building small interactive map tools inside notebooks

| Tool | Strength |
|---|---|
| Folium | Display and presentation |
| ipyleaflet | Interaction and event handling |


## Additional Tool: ipywidgets

`ipyleaflet` is built on top of **ipywidgets**, which provides interactive controls in Jupyter.

Widgets allow us to create interface elements such as:

- buttons
- output displays
- layout controls

In this example we use widgets to:

- control the size of the map
- create a **Clear Saved Points** button
- display status messages


## Install packages if needed

Run this cell first if your environment is missing `ipyleaflet` or `ipywidgets`.


In [1]:
%pip install ipyleaflet ipywidgets

Note: you may need to restart the kernel to use updated packages.


## The Complete Example


In [2]:
import json
from pathlib import Path
from IPython.display import display
from ipyleaflet import GeoJSON, Map, Marker, LayersControl, WidgetControl
import ipywidgets as widgets


OUTFILE = Path("../data/clicked_points.json")
clicked_points = []
markers = []

if OUTFILE.exists():
    clicked_points = json.loads(OUTFILE.read_text())

m = Map(
    center=(40.0, -99.0),
    zoom=5,
    layout=widgets.Layout(width='100%', height='700px')
)
m.add(LayersControl())


def save_points():
    OUTFILE.write_text(json.dumps(clicked_points, indent=2))

# Restore prior markers
for pt in clicked_points:
    marker = Marker(location=(pt["lat"], pt["lon"]))
    markers.append(marker)
    m.add(marker)

output = widgets.Output()
clear_btn = widgets.Button(description="Clear Saved Points")

def handle_interaction(**kwargs):
    if kwargs.get("type") == "click":
        lat, lon = kwargs["coordinates"]

        point = {"lat": round(lat, 6), "lon": round(lon, 6)}
        clicked_points.append(point)

        marker = Marker(location=(point["lat"], point["lon"]))
        markers.append(marker)
        m.add(marker)

        save_points()

        with output:
            print(f"Saved click: {point}")

def clear_points(_):
    clicked_points.clear()
    save_points()

    for marker in markers:
        try:
            m.remove(marker)
        except Exception:
            pass
    markers.clear()

    with output:
        print("Cleared saved points.")

clear_btn.on_click(clear_points)
m.on_interaction(handle_interaction)

m.add(WidgetControl(widget=clear_btn, position="topright"))

display(m, output)

Map(center=[40.0, -99.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

Output()

## Step-by-Step Explanation

### 1. Import Required Libraries

```python
import json
from pathlib import Path
from ipyleaflet import Map, Marker, LayersControl, WidgetControl
import ipywidgets as widgets
```

These libraries allow us to:

- build the map
- place markers
- create widgets
- save data to files

### 2. Define Storage for Clicked Points

```python
OUTFILE = Path("clicked_points.json")
clicked_points = []
markers = []
```

We store:

- coordinates in a Python list
- map markers in a separate list
- the file path where coordinates are saved

### 3. Load Previously Saved Points

```python
if OUTFILE.exists():
    clicked_points = json.loads(OUTFILE.read_text())
```

If the file already exists, we reload earlier clicks so they persist across notebook runs.

### 4. Create the Map

```python
m = Map(
    center=(40.0, -99.0),
    zoom=5,
    layout=widgets.Layout(width='100%', height='700px')
)
```

Unlike Folium, ipyleaflet uses widget layout objects to control width and height.

### 5. Restore Existing Markers

```python
for pt in clicked_points:
    marker = Marker(location=(pt["lat"], pt["lon"]))
    markers.append(marker)
    m.add(marker)
```

### 6. Create Widgets

```python
output = widgets.Output()
clear_btn = widgets.Button(description="Clear Saved Points")
```

### 7. Handle Map Click Events

When a click occurs, we extract coordinates, save them, add a marker, and write the data to disk.

### 8. Clear Saved Points

The clear button deletes stored points, removes markers, rewrites the file, and prints a message.

### 9. Place the Button on the Map

```python
m.add(WidgetControl(widget=clear_btn, position="topright"))
```

### 10. Display the Map and Output Area

```python
display(m, output)
```


## Key Concepts Learned

This example introduces several important ideas:

- **Event-driven programming**
- **Persistent data storage**
- **Widget-based interfaces**
- **Interactive spatial tools**


## Summary

Using **ipyleaflet** and **ipywidgets**, we transformed a static map into an interactive spatial tool.

Users can now:

- click the map
- generate spatial data
- save the results
- manipulate the map with interface controls

These tasks play a big part in the **Missile Geometry 101** final project.


## Altering Marker Style

Changing the way things look is always useful.

For this next quick lesson, change the style of each marker placed on the map.

The example below uses an icon file from the `data` folder:

```python
from ipyleaflet import Marker, Icon

icon = "../data/flag_icon.svg"
icon = Icon(icon_url=icon, icon_size=[30, 30])
marker = Marker(location=[40, -90], icon=icon)
```


## Your Code Here

This version copies the earlier click-capture map and uses the custom flag icon for each marker.

It also checks whether the icon file exists and prints a message if it does not.


In [3]:
import json
from pathlib import Path
from IPython.display import display
from ipyleaflet import Map, Marker, Icon, LayersControl, WidgetControl
import ipywidgets as widgets

OUTFILE = Path("../data/clicked_points_flag_icon.json")
ICON_PATH = Path("../data/flag_icon.svg")

clicked_points = []
markers = []

if OUTFILE.exists():
    clicked_points = json.loads(OUTFILE.read_text())

m2 = Map(
    center=(40.0, -99.0),
    zoom=5,
    layout=widgets.Layout(width='100%', height='700px')
)
m2.add(LayersControl())

output2 = widgets.Output()
clear_btn2 = widgets.Button(description="Clear Saved Points")

custom_icon = None
if ICON_PATH.exists():
    custom_icon = Icon(icon_url=str(ICON_PATH), icon_size=[30, 30])
else:
    with output2:
        print(f"Icon file not found: {ICON_PATH}")

def save_points2():
    OUTFILE.write_text(json.dumps(clicked_points, indent=2))

# Restore previous markers
for pt in clicked_points:
    if custom_icon is not None:
        marker = Marker(location=(pt["lat"], pt["lon"]), icon=custom_icon)
    else:
        marker = Marker(location=(pt["lat"], pt["lon"]))
    markers.append(marker)
    m2.add(marker)

def handle_interaction2(**kwargs):
    if kwargs.get("type") == "click":
        lat, lon = kwargs["coordinates"]

        point = {"lat": round(lat, 6), "lon": round(lon, 6)}
        clicked_points.append(point)

        if custom_icon is not None:
            marker = Marker(location=(point["lat"], point["lon"]), icon=custom_icon)
        else:
            marker = Marker(location=(point["lat"], point["lon"]))

        markers.append(marker)
        m2.add(marker)

        save_points2()

        with output2:
            print(f"Saved click with custom marker: {point}")

def clear_points2(_):
    clicked_points.clear()
    save_points2()

    for marker in markers:
        try:
            m2.remove(marker)
        except Exception:
            pass
    markers.clear()

    with output2:
        print("Cleared saved points.")

clear_btn2.on_click(clear_points2)
m2.on_interaction(handle_interaction2)
m2.add(WidgetControl(widget=clear_btn2, position="topright"))

display(m2, output2)

Map(center=[40.0, -99.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

Output()